# OCTA Classifier — Grid Search


In [12]:
import sys
import logging
from pathlib import Path
import torch

PROJECT_ROOT = Path().resolve()                         
sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT    = PROJECT_ROOT.parent / "data"              
EXCEL_PATH   = DATA_ROOT / "master_excels" / "master_table.xlsx"
ENCODER_PATH = PROJECT_ROOT.parent / "encoders" / "results" / "phase2" / "grl" / "pretrained" / "final_encoder.pth"
RESULTS      = PROJECT_ROOT / "results"

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

print(f"Device      : {device}")
print(f"Project     : {PROJECT_ROOT}")
print(f"Data root   : {DATA_ROOT}")
print(f"Excel       : {EXCEL_PATH}")
print(f"Encoder     : {ENCODER_PATH}")
print(f"Excel exists: {EXCEL_PATH.exists()}")
print(f"Encoder exists: {ENCODER_PATH.exists()}")

Device      : cuda
Project     : C:\Users\klara\Desktop\Codes\03_Model_Training\classifier
Data root   : C:\Users\klara\Desktop\Codes\03_Model_Training\data
Excel       : C:\Users\klara\Desktop\Codes\03_Model_Training\data\master_excels\master_table.xlsx
Encoder     : C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase2\grl\pretrained\final_encoder.pth
Excel exists: True
Encoder exists: True


In [13]:
from scripts.training import run_grid_search, TrainConfig

GRID = {
    "unfreeze_last_n_blocks" : [0, 2, 4],
    "lr"                     : [3e-4, 1e-4, 3e-5],
    "hidden_dims"            : [[512], [512, 256], [512, 512, 256], [256, 128]],
    "dropout"                : [0.3, 0.4, 0.5],
    "label_smoothing"        : [0.0, 0.05, 0.10],
    "encoder_mode"           : ["cls", "cls_mean", "mean_patch"],
    "modality_dropout_prob"  : [0.0, 0.2, 0.3, 0.4],
}

N_RUNS = 300

BASE_CFG = TrainConfig(
    epochs        = 60,
    batch_size    = 32,
    weight_decay  = 1e-4,
    warmup_epochs = 10,
    min_lr        = 1e-6,
    grad_clip     = 1.0,
    use_amp       = True,
    class_weights = "auto",
    encoder_lr_multiplier = 0.05,
    use_bn        = True,
    tf_num_heads  = 4,
    tf_num_layers = 1,
    tf_dropout    = 0.1,
    num_workers   = 4,
    seed          = 42,
)

# Všetky dostupné dáta

In [14]:
run_grid_search(
    experiment_name = "full",
    grid_config     = GRID,
    encoder_path    = ENCODER_PATH,
    excel_path      = EXCEL_PATH,
    data_root       = DATA_ROOT,
    results_root    = RESULTS / "grid_search",
    n_runs          = N_RUNS,
    base_cfg        = BASE_CFG,
    device          = device,
)

2026-05-13 21:31:58,810 [INFO] Fusion svp_only: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:32:03,615 [INFO] Fusion concat: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:32:14,924 [INFO] Fusion gate: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:32:19,772 [INFO] Fusion transformer: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:32:24,608 [INFO] Fusion mean: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:32:29,347 [INFO] Fusion product: nových kombinácií=3888 | naplánovaných=50



GRID SEARCH — experiment=full
encoder_path    : C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase2\grl\pretrained\final_encoder.pth
n_runs          : 300 (50 per fusion)
nových runov    : 300
output          : C:\Users\klara\Desktop\Codes\03_Model_Training\classifier\results\grid_search\full


[1/300] run_136 | fusion=mean | unfreeze_last_n_blocks=4 | lr=0.0003 | hidden_dims=[512, 256] | dropout=0.3 | label_smoothing=0.05 | encoder_mode=cls_mean | modality_dropout_prob=0.2


2026-05-13 21:32:30,029 [INFO] Experiment 'full' | use_col=use_for_cls_full | split_col=split_full | train=1759 | val=386 | test=358
2026-05-13 21:32:30,061 [INFO] OctaClsDataset: 1759 samples (skipped=0, is_train=True)
2026-05-13 21:32:30,067 [INFO] OctaClsDataset: 386 samples (skipped=0, is_train=False)
2026-05-13 21:32:30,076 [INFO] OctaClsDataset: 358 samples (skipped=0, is_train=False)
2026-05-13 21:32:30,221 [INFO] Encoder načítaný (všetko zmrazené): C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase2\grl\pretrained\final_encoder.pth
2026-05-13 21:32:30,377 [INFO] Encoder: unfreeze_last_n=4 (bloky 8–11 + norm) | trainable=7,098,624 | frozen=14,371,200
2026-05-13 21:32:30,377 [INFO] OctaClassifier | fusion=mean | encoder_mode=cls_mean | fused_dim=384 | hidden_dims=[512, 256]
2026-05-13 21:32:30,386 [INFO] Class weights: AMD=3.405, DR=0.113, Healthy=0.190, RVO=0.291
2026-05-13 21:32:30,388 [INFO] Optimizer: cls_lr=3.00e-04 | enc_lr=1.50e-05
C:\Users\klara\Desktop

# SVP DCP 

In [15]:
run_grid_search(
    experiment_name = "svp_dcp",
    grid_config     = GRID,
    encoder_path    = ENCODER_PATH,
    excel_path      = EXCEL_PATH,
    data_root       = DATA_ROOT,
    results_root    = RESULTS / "grid_search",
    n_runs          = N_RUNS,
    base_cfg        = BASE_CFG,
    device          = device,
)

2026-05-13 21:32:59,923 [INFO] Fusion svp_only: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:00,204 [INFO] Fusion concat: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:00,516 [INFO] Fusion gate: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:00,812 [INFO] Fusion transformer: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:01,086 [INFO] Fusion mean: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:01,377 [INFO] Fusion product: nových kombinácií=3888 | naplánovaných=50



GRID SEARCH — experiment=svp_dcp
encoder_path    : C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase2\grl\pretrained\final_encoder.pth
n_runs          : 300 (50 per fusion)
nových runov    : 300
output          : C:\Users\klara\Desktop\Codes\03_Model_Training\classifier\results\grid_search\svp_dcp


[1/300] run_002 | fusion=mean | unfreeze_last_n_blocks=4 | lr=0.0003 | hidden_dims=[512, 256] | dropout=0.3 | label_smoothing=0.05 | encoder_mode=cls_mean | modality_dropout_prob=0.2


2026-05-13 21:33:01,901 [INFO] Experiment 'svp_dcp' | use_col=use_for_cls_svp_dcp | split_col=split_svp_dcp | train=818 | val=185 | test=151
2026-05-13 21:33:01,919 [INFO] OctaClsDataset: 818 samples (skipped=0, is_train=True)
2026-05-13 21:33:01,929 [INFO] OctaClsDataset: 185 samples (skipped=0, is_train=False)
2026-05-13 21:33:01,933 [INFO] OctaClsDataset: 151 samples (skipped=0, is_train=False)
2026-05-13 21:33:02,076 [INFO] Encoder načítaný (všetko zmrazené): C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase2\grl\pretrained\final_encoder.pth
2026-05-13 21:33:02,227 [INFO] Encoder: unfreeze_last_n=4 (bloky 8–11 + norm) | trainable=7,098,624 | frozen=14,371,200
2026-05-13 21:33:02,227 [INFO] OctaClassifier | fusion=mean | encoder_mode=cls_mean | fused_dim=384 | hidden_dims=[512, 256]
2026-05-13 21:33:02,241 [INFO] Class weights: AMD=2.477, DR=0.359, Healthy=0.138, RVO=1.026
2026-05-13 21:33:02,241 [INFO] Optimizer: cls_lr=3.00e-04 | enc_lr=1.50e-05
C:\Users\klara\

KeyboardInterrupt: 

# SVP + DCP vyvážené

In [16]:
run_grid_search(
    experiment_name = "svp_dcp_ballanced",
    grid_config     = GRID,
    encoder_path    = ENCODER_PATH,
    excel_path      = EXCEL_PATH,
    data_root       = DATA_ROOT,
    results_root    = RESULTS / "grid_search",
    n_runs          = N_RUNS,
    base_cfg        = BASE_CFG,
    device          = device,
)

2026-05-13 21:33:13,207 [INFO] Fusion svp_only: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:18,397 [INFO] Fusion concat: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:23,654 [INFO] Fusion gate: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:30,077 [INFO] Fusion transformer: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:36,421 [INFO] Fusion mean: nových kombinácií=3888 | naplánovaných=50
2026-05-13 21:33:42,622 [INFO] Fusion product: nových kombinácií=3888 | naplánovaných=50



GRID SEARCH — experiment=svp_dcp_ballanced
encoder_path    : C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase2\grl\pretrained\final_encoder.pth
n_runs          : 300 (50 per fusion)
nových runov    : 300
output          : C:\Users\klara\Desktop\Codes\03_Model_Training\classifier\results\grid_search\svp_dcp_ballanced


[1/300] run_165 | fusion=mean | unfreeze_last_n_blocks=4 | lr=0.0003 | hidden_dims=[512, 256] | dropout=0.3 | label_smoothing=0.05 | encoder_mode=cls_mean | modality_dropout_prob=0.2


2026-05-13 21:33:43,267 [INFO] Experiment 'svp_dcp_ballanced' | use_col=use_for_cls_svp_dcp_ballanced | split_col=split_svp_dcp_ballanced | train=477 | val=106 | test=102
2026-05-13 21:33:43,276 [INFO] OctaClsDataset: 477 samples (skipped=0, is_train=True)
2026-05-13 21:33:43,282 [INFO] OctaClsDataset: 106 samples (skipped=0, is_train=False)
2026-05-13 21:33:43,287 [INFO] OctaClsDataset: 102 samples (skipped=0, is_train=False)
2026-05-13 21:33:43,457 [INFO] Encoder načítaný (všetko zmrazené): C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase2\grl\pretrained\final_encoder.pth
2026-05-13 21:33:43,622 [INFO] Encoder: unfreeze_last_n=4 (bloky 8–11 + norm) | trainable=7,098,624 | frozen=14,371,200
2026-05-13 21:33:43,624 [INFO] OctaClassifier | fusion=mean | encoder_mode=cls_mean | fused_dim=384 | hidden_dims=[512, 256]
2026-05-13 21:33:43,637 [INFO] Class weights: AMD=2.323, DR=0.337, Healthy=0.378, RVO=0.962
2026-05-13 21:33:43,640 [INFO] Optimizer: cls_lr=3.00e-04 | e

KeyboardInterrupt: 